### No. 1

In [2]:
import os
import pandas as pd

SOURCE = "archive"
events = pd.read_csv(
    os.path.join(SOURCE, "events.csv"),
    dtype={"transactionid": str},
)
events["datetime"] = pd.to_datetime(events["timestamp"], unit="ms")  # UTC

P1_START, P1_END = "2015-05-26", "2015-07-26"   # training
P2_START, P2_END = "2015-08-04", "2015-08-19"   # testing

def filter_period(df, start, end):
    s = pd.Timestamp(start)
    e = pd.Timestamp(end) + pd.Timedelta(days=1)   # akhir inklusif
    return df[(df["datetime"] >= s) & (df["datetime"] < e)]

train = filter_period(events, P1_START, P1_END)
test = filter_period(events, P2_START, P2_END)

os.makedirs("data", exist_ok=True)
cols = ["timestamp", "visitorid", "event", "itemid", "transactionid"]
train[cols].to_csv("data/events_train.csv", index=False)
test[cols].to_csv("data/events_test.csv", index=False)

### No. 2

In [3]:
events.info()
events["event"].value_counts()
events["transactionid"].notna().sum()   # harus sama dengan jumlah event 'transaction'

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2756101 entries, 0 to 2756100
Data columns (total 6 columns):
 #   Column         Dtype         
---  ------         -----         
 0   timestamp      int64         
 1   visitorid      int64         
 2   event          object        
 3   itemid         int64         
 4   transactionid  object        
 5   datetime       datetime64[ns]
dtypes: datetime64[ns](1), int64(3), object(2)
memory usage: 126.2+ MB


np.int64(22457)

### No. 3

In [4]:
from neo4j import GraphDatabase

URI = "neo4j://127.0.0.1:7687"
USER, PASSWORD = "neo4j", "badainamko"

driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))
session = driver.session(database="system")
print(session.run("SHOW DATABASES YIELD name RETURN name").values())
session.close()
driver.close()

[['neo4j'], ['system']]


In [5]:
import argparse, getpass

def parse_arguments(argv=None):
    parser = argparse.ArgumentParser(
        description="Impor dataset RetailRocket (events) ke database Neo4j."
    )
    parser.add_argument("-u", "--user", default="neo4j",
                        help="Username Neo4j (default: neo4j)")
    parser.add_argument("-p", "--password", default=None,
                        help="Password Neo4j (jika kosong, akan ditanya)")
    parser.add_argument("-b", "--bolt", default="neo4j://127.0.0.1:7687",
                        help="Neo4j URI / Bolt (default: neo4j://127.0.0.1:7687)")
    parser.add_argument("-s", "--source", default="data",
                        help="Direktori/folder sumber data (default: data)")
    return parser.parse_args(argv)

args = parse_arguments([])          # [] = pakai nilai default (khusus di notebook)
if args.password is None:
    args.password = getpass.getpass("Password Neo4j: ")

In [6]:
import os, sys

csv_file_path = os.path.join(args.source, "events_train.csv")   # file hasil filter periode lo

if not os.path.exists(csv_file_path):
    print(f"Error: File '{csv_file_path}' tidak ditemukan!")
    sys.exit(1)

try:
    driver = GraphDatabase.driver(args.bolt, auth=(args.user, args.password))
    driver.verify_connectivity()
    print("Koneksi ke Neo4j berhasil.")
except Exception as e:
    print(f"Gagal terhubung ke database: {e}")
    sys.exit(1)

Koneksi ke Neo4j berhasil.


### No. 4

In [7]:
DB = "neo4j"

def create_constraints(session):
    session.run("CREATE CONSTRAINT visitor_id IF NOT EXISTS "
                "FOR (v:Visitor) REQUIRE v.visitorId IS UNIQUE")
    session.run("CREATE CONSTRAINT item_id IF NOT EXISTS "
                "FOR (i:Item) REQUIRE i.itemId IS UNIQUE")

def import_batch(tx, batch):
    cypher_query = """
    UNWIND $batch AS row
    MERGE (v:Visitor {visitorId: row.visitorid})
    MERGE (i:Item {itemId: row.itemid})
    WITH v, i, row
    CALL (v, i, row) {
        WITH v, i, row WHERE row.event = 'view'
        CREATE (v)-[:VIEWED {timestamp: row.timestamp}]->(i)
        UNION ALL
        WITH v, i, row WHERE row.event = 'addtocart'
        CREATE (v)-[:ADDED_TO_CART {timestamp: row.timestamp}]->(i)
        UNION ALL
        WITH v, i, row WHERE row.event = 'transaction'
        CREATE (v)-[:BOUGHT {timestamp: row.timestamp,
                             transactionId: row.transactionid}]->(i)
    }
    """
    tx.run(cypher_query, batch=batch)

In [8]:
driver.execute_query("MATCH (n) DETACH DELETE n", database_=DB)

EagerResult(records=[], summary=<neo4j._work.summary.ResultSummary object at 0x000001B3E6EA2620>, keys=[])

In [9]:
import pandas as pd
from tqdm import tqdm

batch_size = 5000

with driver.session(database=DB) as session:
    create_constraints(session)

    print(f"Membaca file data dari: {csv_file_path}")
    total_rows = sum(1 for _ in open(csv_file_path)) - 1      # kurangi baris header
    total_chunks = (total_rows // batch_size) + 1
    print(f"Total baris: {total_rows:,} | Total batch: {total_chunks}")

    chunks = pd.read_csv(csv_file_path, chunksize=batch_size,
                         dtype={"transactionid": str})

    print("Memulai proses impor data ke Neo4j...")
    for chunk in tqdm(chunks, total=total_chunks, desc="Mengimpor Batch Data"):
        # Membersihkan data null/NaN pada transactionid
        chunk["transactionid"] = chunk["transactionid"].fillna("").astype(str)
        records = chunk.to_dict(orient="records")
        session.execute_write(import_batch, records)

driver.close()
print("Proses impor berhasil.")

Membaca file data dari: data\events_train.csv
Total baris: 1,312,675 | Total batch: 263
Memulai proses impor data ke Neo4j...


Mengimpor Batch Data: 100%|██████████| 263/263 [02:36<00:00,  1.68it/s]

Proses impor berhasil.


In [10]:
driver = GraphDatabase.driver(args.bolt, auth=(args.user, args.password))
r, _, _ = driver.execute_query(
    "MATCH (n) RETURN labels(n)[0] AS label, count(*) AS jumlah", database_=DB)
print([dict(x) for x in r])
r, _, _ = driver.execute_query(
    "MATCH ()-[r]->() RETURN type(r) AS tipe, count(*) AS jumlah", database_=DB)
print([dict(x) for x in r])

[{'label': 'Visitor', 'jumlah': 677687}, {'label': 'Item', 'jumlah': 170744}]
[{'tipe': 'VIEWED', 'jumlah': 1269759}, {'tipe': 'ADDED_TO_CART', 'jumlah': 32263}, {'tipe': 'BOUGHT', 'jumlah': 10653}]


### No. 5

In [11]:
def import_batch(tx, batch):
    cypher_query = """
    UNWIND $batch AS row
    MERGE (v:Visitor {visitorId: row.visitorid})
    MERGE (i:Item {itemId: row.itemid})

    WITH v, i, row
    CALL (v, i, row) {
        WITH v, i, row WHERE row.event = 'view'
        CREATE (v)-[:VIEWED {timestamp: row.timestamp}]->(i)

        UNION ALL

        WITH v, i, row WHERE row.event = 'addtocart'
        CREATE (v)-[:ADDED_TO_CART {timestamp: row.timestamp}]->(i)

        UNION ALL

        WITH v, i, row WHERE row.event = 'transaction'
        CREATE (v)-[:BOUGHT {timestamp: row.timestamp,
                             transactionId: row.transactionid}]->(i)
    }
    """
    tx.run(cypher_query, batch=batch)

In [12]:
import pandas as pd
from neo4j import GraphDatabase

driver = GraphDatabase.driver(args.bolt, auth=(args.user, args.password))

sample = pd.read_csv(csv_file_path, nrows=5, dtype={"transactionid": str})
sample["transactionid"] = sample["transactionid"].fillna("").astype(str)
records = sample.to_dict(orient="records")
print(sample[["visitorid", "event", "itemid"]])

with driver.session(database=DB) as session:
    tx = session.begin_transaction()
    before = tx.run("MATCH ()-[r]->() RETURN count(r) AS c").single()["c"]
    import_batch(tx, records)
    after = tx.run("MATCH ()-[r]->() RETURN count(r) AS c").single()["c"]
    print(f"Relasi sebelum: {before:,} | sesudah (di dalam transaksi): {after:,} | selisih: {after - before}")
    tx.rollback()          # dibatalkan, data asli tidak berubah

driver.close()

   visitorid event  itemid
0     257597  view  355908
1     992329  view  248676
2     111016  view  318965
3     483717  view  253185
4     951259  view  367447
Relasi sebelum: 1,312,675 | sesudah (di dalam transaksi): 1,312,680 | selisih: 5


# 1.3 Impor Data RetailRocket ke Database Neo4j

**Dataset:** RetailRocket e-commerce (Roman Zykov, Kaggle)
**Lingkungan:** Python 3.10.20, Neo4j Desktop 2.2.1 (server 2026.09.0), driver `neo4j` 6.4.1
**Periode data (fold No. 26, NIM 23031554147):**

| Fold | Mulai | Hingga |
|---|---|---|
| Training (P1) | 26 Mei 2015 | 26 Juli 2015 |
| Testing (P2) | 4 Agustus 2015 | 19 Agustus 2015 |

---

## 1. Unduh data
Dataset diunduh dari Kaggle (RetailRocket) dan disimpan di folder `archive`. Isinya `events.csv`, `category_tree.csv`, `item_properties_part1.csv`, dan `item_properties_part2.csv`. Yang dipakai untuk UTS hanya `events.csv`.

## 2. Fitur pada `events.csv`
Total data: **2.756.101 baris**.

| Fitur | Arti | Peran di graf |
|---|---|---|
| `timestamp` | Waktu kejadian, unix epoch (milidetik) | Properti `timestamp` pada relasi |
| `visitorid` | ID unik pengunjung | Node `Visitor` (`visitorId`) |
| `event` | Jenis interaksi: `view`, `addtocart`, `transaction` | Menentukan tipe relasi |
| `itemid` | ID unik produk | Node `Item` (`itemId`) |
| `transactionid` | ID transaksi, hanya terisi saat event `transaction` | Properti `transactionId` pada relasi `BOUGHT` |

### Persiapan data: filter periode
Timestamp dikonversi ke datetime (UTC), lalu data difilter sesuai periode fold. Tanggal akhir bersifat **inklusif** (sampai 23:59:59 UTC). Hasilnya disimpan ke CSV:

- `data/events_train.csv` (P1: 2015-05-26 s.d. 2015-07-26), **1.312.675 baris**
- `data/events_test.csv` (P2: 2015-08-04 s.d. 2015-08-19), `[ISI: jumlah baris]`

## 3. Mekanisme session Neo4j
- **Driver** adalah koneksi utama (dibuat sekali, dipakai bersama), sedangkan **session** adalah jalur kerja ringan ke satu database.
- Pada contoh dasar, `driver.session(database="system")` dipakai untuk administrasi. Untuk data RetailRocket dipakai database `neo4j`.
- Versi `argparse` memuat parameter `--user`, `--password`, `--bolt`, dan `--source` agar skrip dapat dijalankan dengan konfigurasi berbeda. Password tidak ditulis di kode, melainkan diminta lewat `getpass`.
- Sebelum impor dilakukan pengecekan keberadaan file CSV dan konektivitas (`driver.verify_connectivity()`). Hasil: **koneksi ke Neo4j berhasil**.

**Penyesuaian:** file sumber memakai `events_train.csv` (hasil filter periode), bukan `events.csv`.

## 4. Impor dengan mekanisme batch
Data dibaca dan dikirim per potongan agar hemat memori dan setiap transaksi tetap ringan.

| Komponen | Fungsi |
|---|---|
| `batch_size = 5000` | Jumlah baris per batch |
| `create_constraints` | Constraint unik `Visitor.visitorId` dan `Item.itemId` (otomatis membuat index sehingga `MERGE` cepat) |
| `pd.read_csv(chunksize=...)` | Membaca CSV bertahap. `dtype={"transactionid": str}` mencegah ID terbaca sebagai `"4000.0"` |
| `total_rows`, `total_chunks` | Dasar progress bar `tqdm` |
| `fillna("").astype(str)` | Membersihkan nilai kosong pada `transactionid` |
| `chunk.to_dict("records")` | Mengubah batch menjadi daftar dictionary untuk parameter Cypher |
| `session.execute_write(import_batch, records)` | Transaksi tulis terkelola (retry otomatis jika gagal sementara) |

Hasil: **1.312.675 baris, 263 batch, selesai sekitar 2 menit 36 detik**.

## 5. Cypher untuk impor (`import_batch`)

| Bagian query | Penjelasan |
|---|---|
| `UNWIND $batch AS row` | Memecah daftar parameter `$batch` menjadi baris-baris (satu `row` = satu event) |
| `MERGE (v:Visitor {visitorId: row.visitorid})` | Membuat node Visitor hanya jika belum ada (get-or-create) |
| `MERGE (i:Item {itemId: row.itemid})` | Sama, untuk node Item |
| `CALL (v, i, row) { ... UNION ALL ... }` | Subquery percabangan: tiap cabang memfilter `row.event` lalu membuat relasi yang sesuai |
| `CREATE (v)-[:VIEWED {timestamp}]->(i)` | Event `view` |
| `CREATE (v)-[:ADDED_TO_CART {timestamp}]->(i)` | Event `addtocart` |
| `CREATE (v)-[:BOUGHT {timestamp, transactionId}]->(i)` | Event `transaction` |

**Mengapa `MERGE` untuk node dan `CREATE` untuk relasi?** Node harus unik (satu pengunjung atau produk = satu node). Relasi adalah **kejadian**: visitor yang sama bisa melihat item yang sama berkali-kali, dan semuanya harus tercatat. Konsekuensinya graf ini adalah **multigraf** (boleh ada banyak relasi antara pasangan visitor-item yang sama), dan impor tidak boleh diulang tanpa reset.

**Penyesuaian sintaks:** PDF memakai `CALL { WITH v, i, row ... }` (sintaks lama, *importing WITH*). Di sini dipakai `CALL (v, i, row) { ... }` (*variable scope clause*) yang sesuai dengan Neo4j versi baru. Logikanya identik.

## Hasil impor

| Node | Jumlah |
|---|---|
| `Visitor` | 677.687 |
| `Item` | 170.744 |

| Relasi | Jumlah |
|---|---|
| `VIEWED` | 1.269.759 |
| `ADDED_TO_CART` | 32.263 |
| `BOUGHT` | 10.653 |
| **Total** | **1.312.675** |

Total relasi sama dengan jumlah baris `events_train.csv` (setiap baris menghasilkan tepat satu relasi), sehingga tidak ada data yang hilang atau ganda.

### Uji fungsi `import_batch` (dengan rollback)
Fungsi diuji pada 5 baris contoh dalam satu transaksi yang kemudian dibatalkan:

```
Relasi sebelum: 1,312,675 | sesudah (di dalam transaksi): 1,312,680 | selisih: 5
```
Selisih 5 sesuai dengan 5 baris masukan. Transaksi di-*rollback* sehingga data asli tetap 1.312.675 relasi.

### Skema graf hasil impor
```
(:Visitor {visitorId}) -[:VIEWED {timestamp}]->        (:Item {itemId})
(:Visitor {visitorId}) -[:ADDED_TO_CART {timestamp}]-> (:Item {itemId})
(:Visitor {visitorId}) -[:BOUGHT {timestamp, transactionId}]-> (:Item {itemId})
```
Graf **bipartit berarah** (dua himpunan node, Visitor dan Item, dengan relasi hanya antar himpunan) dan berupa multigraf.